# Molecular dynamics of a protein in water with GROMACS

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yboulaamane/cadd-labs/blob/main/MD-simulation-Gromacs.ipynb)

A crystal structure is a single frozen snapshot. Molecular dynamics sets the atoms moving under
a force field so you can watch how a protein actually behaves in water. This notebook runs a full
[GROMACS](https://www.gromacs.org/) simulation of hen egg-white lysozyme (PDB
[1AKI](https://www.rcsb.org/structure/1AKI)) end to end, then checks whether the fold stays stable.

**What you'll do**

- Install GROMACS in one cell and prepare the protein in a box of water with ions.
- Minimise, equilibrate (NVT then NPT), and run a short production trajectory.
- Plot the temperature, pressure, density, backbone RMSD and radius of gyration to judge the run.

It follows Justin Lemkul's [lysozyme in water tutorial](http://www.mdtutorials.com/gmx/lysozyme/index.html),
which explains every step in depth, using the AMBER99SB force field and SPC/E water.

Runtime: choose a **GPU** (Runtime -> Change runtime type); it also runs on CPU, much more slowly.
Colab deletes files when a session ends, so copy the results to Google Drive at the end if you want to keep them.

## 1. Install GROMACS

GROMACS comes from [conda-forge](https://anaconda.org/conda-forge/gromacs), installed with
micromamba into its own folder, so Colab's Python is left alone. On a GPU runtime this
is the CUDA build, which brings its own CUDA libraries: a download of several GB. On a
CPU runtime it is the much smaller CPU build.

In [ ]:
%%bash
set -e
if nvidia-smi > /dev/null 2>&1; then build="nompi_cuda*"; else build="nompi_h*"; fi
curl -fsSL -o /usr/local/bin/micromamba \
    https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64
chmod +x /usr/local/bin/micromamba
micromamba create -y -q -p /content/gromacs -c conda-forge "gromacs=2026.3=$build"

In [ ]:
import os
import re
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({"figure.dpi": 120, "axes.titleweight": "bold", "savefig.bbox": "tight"})

os.environ["PATH"] = "/content/gromacs/bin:" + os.environ["PATH"]
os.makedirs("/content/lysozyme", exist_ok=True)
os.chdir("/content/lysozyme")
!gmx --version 2>&1 | grep -E "GROMACS version|GPU support"

In [ ]:
def gmx(command, answers=None):
    """Run one GROMACS command. Answers to its group prompts go in `answers`, one per line.
    The full output is appended to gmx.out; if the command fails, its last lines are shown."""
    result = subprocess.run(f"gmx {command}", shell=True, input=answers, text=True, capture_output=True)
    with open("gmx.out", "a") as log:
        log.write(f"$ gmx {command}\n{result.stdout}{result.stderr}\n")
    if result.returncode != 0:
        tail = "\n".join((result.stdout + result.stderr).splitlines()[-25:])
        raise RuntimeError(f"gmx {command.split()[0]} failed:\n{tail}")


def molecules(top="topol.top"):
    """The [ molecules ] section of a topology: what the system contains."""
    text = open(top).read()
    print(text[text.index("[ molecules ]"):].strip())


def plot_xvg(path, title=None):
    """Plot every data column of a GROMACS .xvg file against the first."""
    labels, legends = {}, []
    for line in open(path):
        if m := re.match(r'@\s+([xy])axis\s+label\s+"(.*)"', line):
            labels[m.group(1)] = m.group(2)
        elif m := re.match(r'@\s+s\d+\s+legend\s+"(.*)"', line):
            legends.append(m.group(1))
    data = np.loadtxt(path, comments=["#", "@"], ndmin=2)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for i in range(1, data.shape[1]):
        ax.plot(data[:, 0], data[:, i], linewidth=1.5,
                label=legends[i - 1] if i - 1 < len(legends) else None)
    ax.set(xlabel=labels.get("x", ""), ylabel=labels.get("y", ""), title=title or path)
    if legends:
        ax.legend(frameon=True)
    plt.tight_layout()
    plt.show()

## 2. Prepare the protein

Download the crystal structure and remove the crystal waters (residue HOH). `pdb2gmx`
then adds hydrogens and writes the topology (`topol.top`) and position restraints
(`posre.itp`) for the chosen force field and water model.

In [ ]:
!curl -fsSL -o 1AKI.pdb https://files.rcsb.org/download/1AKI.pdb
!grep -v HOH 1AKI.pdb > 1AKI_clean.pdb
gmx("pdb2gmx -f 1AKI_clean.pdb -o processed.gro -ff amber99sb -water spce -ignh")
molecules()

## 3. Box, water and ions

A cubic box with at least 1.0 nm between the protein and its edges, filled with water.
Lysozyme carries a net charge of +8, so `genion` replaces 8 water molecules with
chloride ions to make the system neutral.

In [ ]:
gmx("editconf -f processed.gro -o newbox.gro -c -d 1.0 -bt cubic")
gmx("solvate -cp newbox.gro -cs spc216.gro -o solv.gro -p topol.top")
molecules()

In [ ]:
%%writefile ions.mdp
; Only used to assemble a .tpr for gmx genion; nothing is run with it
integrator      = steep
emtol           = 1000.0
emstep          = 0.01
nsteps          = 50000
nstlist         = 1
cutoff-scheme   = Verlet
coulombtype     = cutoff
rcoulomb        = 1.0
rvdw            = 1.0
pbc             = xyz

In [ ]:
gmx("grompp -f ions.mdp -c solv.gro -p topol.top -o ions.tpr")
gmx("genion -s ions.tpr -o solv_ions.gro -p topol.top -pname NA -nname CL -neutral", answers="SOL\n")
molecules()

## 4. Energy minimisation

Removes clashes left by building the system. The potential energy should drop steeply,
then level off at a large negative value.

In [ ]:
%%writefile minim.mdp
; Energy minimisation: steepest descent until the largest force is below 1000 kJ/mol/nm
integrator      = steep
emtol           = 1000.0
emstep          = 0.01
nsteps          = 50000
nstlist         = 1
cutoff-scheme   = Verlet
coulombtype     = PME
rcoulomb        = 1.0
rvdw            = 1.0
pbc             = xyz

In [ ]:
gmx("grompp -f minim.mdp -c solv_ions.gro -p topol.top -o em.tpr")
gmx("mdrun -deffnm em")
!grep -E "converged|Potential Energy|Maximum force" em.log

In [ ]:
gmx("energy -f em.edr -o potential.xvg", answers="Potential\n")
plot_xvg("potential.xvg", "Energy minimisation")

## 5. Equilibration

Two 100 ps runs with the protein's heavy atoms restrained, so the water and ions settle
around it: first at constant volume to reach 300 K (NVT), then at constant pressure to
reach 1 bar and the right density (NPT).

In [ ]:
%%writefile nvt.mdp
; NVT equilibration: 100 ps at 300 K, protein heavy atoms restrained
define                  = -DPOSRES
integrator              = md
nsteps                  = 50000     ; 2 fs * 50000 = 100 ps
dt                      = 0.002
nstenergy               = 500
nstlog                  = 500
nstxout-compressed      = 500
continuation            = no
constraint_algorithm    = lincs
constraints             = h-bonds
lincs_iter              = 1
lincs_order             = 4
cutoff-scheme           = Verlet
nstlist                 = 10
rcoulomb                = 1.0
rvdw                    = 1.0
DispCorr                = EnerPres
coulombtype             = PME
pme_order               = 4
fourierspacing          = 0.16
tcoupl                  = V-rescale
tc-grps                 = Protein Non-Protein
tau_t                   = 0.1     0.1
ref_t                   = 300     300
pcoupl                  = no
pbc                     = xyz
gen_vel                 = yes
gen_temp                = 300
gen_seed                = -1

In [ ]:
gmx("grompp -f nvt.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr")
gmx("mdrun -deffnm nvt")
!grep -B 1 "Performance:" nvt.log

In [ ]:
gmx("energy -f nvt.edr -o temperature.xvg", answers="Temperature\n")
plot_xvg("temperature.xvg", "NVT: temperature")

In [ ]:
%%writefile npt.mdp
; NPT equilibration: 100 ps at 300 K and 1 bar, protein heavy atoms restrained
define                  = -DPOSRES
integrator              = md
nsteps                  = 50000     ; 2 fs * 50000 = 100 ps
dt                      = 0.002
nstenergy               = 500
nstlog                  = 500
nstxout-compressed      = 500
continuation            = yes
constraint_algorithm    = lincs
constraints             = h-bonds
lincs_iter              = 1
lincs_order             = 4
cutoff-scheme           = Verlet
nstlist                 = 10
rcoulomb                = 1.0
rvdw                    = 1.0
DispCorr                = EnerPres
coulombtype             = PME
pme_order               = 4
fourierspacing          = 0.16
tcoupl                  = V-rescale
tc-grps                 = Protein Non-Protein
tau_t                   = 0.1     0.1
ref_t                   = 300     300
pcoupl                  = C-rescale
pcoupltype              = isotropic
tau_p                   = 1.0
ref_p                   = 1.0
compressibility         = 4.5e-5
refcoord_scaling        = com
pbc                     = xyz
gen_vel                 = no

In [ ]:
gmx("grompp -f npt.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr")
gmx("mdrun -deffnm npt")
!grep -B 1 "Performance:" npt.log

In [ ]:
gmx("energy -f npt.edr -o pressure.xvg", answers="Pressure\n")
plot_xvg("pressure.xvg", "NPT: pressure")

In [ ]:
gmx("energy -f npt.edr -o density.xvg", answers="Density\n")
plot_xvg("density.xvg", "NPT: density")

The pressure fluctuates by hundreds of bar around 1 bar; that is normal for a system this
small. The density should be stable, close to 1000 kg/m&sup3;.

## 6. Production run

1 ns without restraints. On a Colab GPU this is the longest step; the performance line
below reports the speed in ns/day. For a longer run, raise `nsteps` in `md.mdp`.

In [ ]:
%%writefile md.mdp
; Production: 1 ns at 300 K and 1 bar, no restraints
integrator              = md
nsteps                  = 500000    ; 2 fs * 500000 = 1 ns
dt                      = 0.002
nstenergy               = 5000
nstlog                  = 5000
nstxout-compressed      = 5000      ; one frame every 10 ps
continuation            = yes
constraint_algorithm    = lincs
constraints             = h-bonds
lincs_iter              = 1
lincs_order             = 4
cutoff-scheme           = Verlet
nstlist                 = 10
rcoulomb                = 1.0
rvdw                    = 1.0
DispCorr                = EnerPres
coulombtype             = PME
pme_order               = 4
fourierspacing          = 0.16
tcoupl                  = V-rescale
tc-grps                 = Protein Non-Protein
tau_t                   = 0.1     0.1
ref_t                   = 300     300
pcoupl                  = Parrinello-Rahman
pcoupltype              = isotropic
tau_p                   = 2.0
ref_p                   = 1.0
compressibility         = 4.5e-5
pbc                     = xyz
gen_vel                 = no

In [ ]:
gmx("grompp -f md.mdp -c npt.gro -t npt.cpt -p topol.top -o md.tpr")
gmx("mdrun -deffnm md")
!grep -B 1 "Performance:" md.log

## 7. Analysis

First make the protein whole and centred in the box in every frame, so it does not
appear split across the periodic boundaries. Groups are chosen by name.

In [ ]:
gmx("trjconv -s md.tpr -f md.xtc -o md_noPBC.xtc -pbc mol -center", answers="Protein\nSystem\n")

Backbone RMSD after fitting to the backbone, relative to the structure at the start of
the production run. A plateau means the structure is stable on this time scale.

In [ ]:
gmx("rms -s md.tpr -f md_noPBC.xtc -o rmsd.xvg -tu ns", answers="Backbone\nBackbone\n")
plot_xvg("rmsd.xvg", "Backbone RMSD")

Radius of gyration of the protein: a measure of its compactness. A folded protein keeps
it roughly constant.
`Rg` is the overall radius; the other three curves are the radii about the x, y and z axes.

In [ ]:
gmx("gyrate -s md.tpr -f md_noPBC.xtc -sel Protein -o gyrate.xvg")
plot_xvg("gyrate.xvg", "Radius of gyration")

## Reading the run

- **Temperature, pressure, density** should settle around their targets (300 K, 1 bar, ~1000 kg/m3)
  during equilibration. Pressure swings by hundreds of bar in a system this small; that is normal.
- **Backbone RMSD** rising then flattening means the fold has relaxed and stayed put: a stable run.
  A curve still climbing at the end means you need a longer simulation.
- **Radius of gyration** staying roughly flat says the protein kept its compact fold rather than
  unfolding.

This is 1 ns, long enough to see the system settle, far too short for real conclusions. Production
studies run tens to hundreds of nanoseconds; raise `nsteps` in `md.mdp` to go longer.

## 8. Keep the results (optional)

Copies the inputs, trajectory and analysis files to `MyDrive/lysozyme_md` on your
Google Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/lysozyme_md
!cp *.mdp topol.top posre.itp md.tpr md.gro md.edr md.log md_noPBC.xtc *.xvg /content/drive/MyDrive/lysozyme_md/